# 097 — Embeddings y búsqueda vectorial

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** `‖q‖ = 3`, `‖a‖ = 6`, `‖b‖ = 3`, `‖c‖ = 3`.
`cos(q,a) = (8+2+8)/(3·6) = 18/18 = 1.000` (a = 2q).
`cos(q,b) = 3/(3·3) = 0.333`. `cos(q,c) = (4−2+2)/(3·3) = 4/9 ≈ 0.444`.
Ranking: **a > c > b**.

**Ejercicio 2.** `q̂ = (2/3, 1/3, 2/3)`, `â = (2/3, 1/3, 2/3)`: idénticos, así que
`q̂·â = 1 = cos(q,a)` y `‖q̂ − â‖² = 0 = 2 − 2·1`. La identidad
`‖u−v‖² = 2 − 2·cos` solo vale con vectores unitarios: por eso los índices normalizan
al ingestar.

**Ejercicio 3.** `cos(q, 10b) = cos(q, b) = 0.333` (invariancia a escala), pero
`‖q − 10b‖ = √(4 + 841 + 4) ≈ 29.1 ≫ ‖q − b‖ = √(4+4+4) ≈ 2.9`. Sin normalizar,
la euclídea castiga la magnitud y el coseno la ignora; para embeddings sin normalizar
se prefiere coseno (o normalizar y usar producto punto), porque la norma suele
correlacionar con longitud del texto, no con su significado.

**Ejercicio 4.** El contrato se verifica en el código: `kind == "retrieval"` y
`evidence` no vacía. El valor interno puede variar con la semilla; el contrato no.


In [ ]:
result = run_lab("retrieval", seed=97)
assert result["kind"] == "retrieval"
assert result["evidence"]
show(result)


In [ ]:
import math

def cos(u, v):
    dot = sum(x*y for x, y in zip(u, v))
    return dot / (math.dist(u, (0,0,0)) * math.dist(v, (0,0,0)))

q = (2, 1, 2); a = (4, 2, 4); b = (0, 3, 0); c = (2, -2, 1)
for name, v in [("a", a), ("b", b), ("c", c)]:
    print(f"cos(q,{name}) = {cos(q, v):.3f}")

# Ejercicio 2: identidad ‖q̂−â‖² = 2 − 2·cos(q,a)
nq = math.dist(q, (0,0,0)); na = math.dist(a, (0,0,0))
qh = [x/nq for x in q]; ah = [x/na for x in a]
lhs = sum((x-y)**2 for x, y in zip(qh, ah))
print("‖q̂−â‖² =", round(lhs, 6), " 2−2cos =", round(2 - 2*cos(q, a), 6))

# Ejercicio 3: escala
b10 = tuple(10*x for x in b)
print("cos(q,10b) =", round(cos(q, b10), 3), " euclídea(q,10b) =", round(math.dist(q, b10), 1))


## Reflexión

1. ¿Por qué `cos(q, 2q) = 1` pero `‖q − 2q‖ ≠ 0`, y qué implica eso para elegir métrica en un índice que no normaliza vectores?
2. Si HNSW devuelve recall@10 = 0.92 frente al baseline flat, ¿qué decisiones del sistema (efSearch, M, tamaño de colección) revisarías antes de aceptar esa pérdida?
3. ¿Qué evidencia necesitarías para afirmar que un modelo de embeddings "funciona" en tu dominio, más allá de su puntuación en un benchmark público?
